# 2단계 — 학습률 (옵티마이저별로 훑어보기)

기준선은 1단계의 He 초기화로 확정했다 (2026-10-29). 이 노트북은 거기서 **학습률 하나만** 바꿔 본다. 다만 적정 학습률은 옵티마이저마다 자릿수가 다르므로, 옵티마이저 3개 각각의 안에서 학습률을 바꾼다.

| | 값 |
|---|---|
| 프레임워크 | Keras, PyTorch |
| 옵티마이저 | Adam · SGD + momentum 0.9 · SGD (momentum 없음). momentum 값은 v1처럼 0.9로 고정 |
| 학습률 | 1e-4, 1e-3, 1e-2, 1e-1 (10배 간격) |
| 시드 | 0 (훑어보기) |
| 실행 횟수 | 3 × 4 × 2 = **24번** |
| 그 밖의 조건 | 1단계와 같다: He 초기화, 2 Conv(32·64) + Dropout 0.5 + Dense(11), batch 32, 5 epoch, 증강 없음, 학습률 고정(스케줄러 없음) |
| 소요 시간 | 대략 84분 (Keras 12번 ≈ 30분, PyTorch 12번 ≈ 54분) |

**이 단계는 옵티마이저끼리 비교하는 단계가 아니다.** 각 옵티마이저 안에서 학습률만 바꾸고, 어느 자릿수가 맞는지 찾는다. 옵티마이저 비교는 다음 단계에서 각자 최적 학습률로 한다. 기준선(Adam · 1e-3)은 이 표 안의 한 칸이다.

**격자를 1e-4~1e-1로 잡은 이유**: 처음 계획은 1e-4~1e-2였다. 그런데 v1에서 SGD(lr 1e-3)는 5에폭 CNN에서 20.5%에 그쳤고(Adam 44.1%) momentum 0.9를 더하면 +12%p였다. SGD 계열의 최적은 1e-2보다 위일 가능성이 커서, 격자가 1e-2에서 끝나면 "클수록 좋다"만 알게 된다. 그래서 1e-1까지 넓히고, 칸 수를 4개로 유지하려고 간격을 10배로 했다. 촘촘한 탐색(3e-4, 3e-3 등)은 최적의 자릿수를 안 다음에 한다.

## 학습률이란

**한 번 갱신할 때 가중치를 기울기 방향으로 얼마나 크게 움직일지 정하는 배율**이다. 배치(32장)마다 "손실을 줄이려면 가중치를 어느 쪽으로 얼마나 바꿔야 하나"(기울기 g)를 구하고, 학습률만큼의 비율로 움직인다. 여기서는 한 에폭이 309번, 5에폭이면 약 1,545번 움직인다.

| 옵티마이저 | 한 번에 움직이는 양 | 특징 |
|---|---|---|
| SGD | lr × g | 기울기에 그대로 비례한다. 기울기가 작으면 조금밖에 안 움직인다 |
| SGD + momentum 0.9 | lr × v (v = 이전 방향을 누적한 속도) | 같은 방향이 이어지면 속도가 쌓여 최대 10배(1 ÷ (1−0.9))까지 커진다. 같은 lr이어도 SGD보다 크게 움직인다 |
| Adam | 대략 lr 안팎 | 기울기를 최근 크기로 나눠 쓰므로 기울기가 크든 작든 한 번에 움직이는 양이 비슷하다 |

그래서 **적정 학습률은 옵티마이저마다 자릿수가 다르다.** v1이 전부 1e-3으로 통일한 게 문제점이었던 이유다 (Adam에게는 기본값, SGD에게는 너무 작다).

## 예상되는 작용

- **너무 작다** — 가중치가 조금씩만 움직여서 5에폭 안에 바닥까지 못 간다. 학습 곡선이 완만하고 val loss가 끝까지 내려가는 중이다
- **알맞다** — 빠르게 내려간다. 다만 이 모델은 2~3에폭 뒤 과적합이 시작된다 (1단계에서 확인). 빨리 닿을수록 과적합도 빨리 시작된다
- **너무 크다** — 최저점 근처에서 튀거나(loss가 출렁), 초반에 가중치가 망가져서(ReLU가 죽거나 NaN) 학습이 되지 않는다. train acc가 찍기 수준(9%)에서 오르지 않는다

크기 감각: He 초기화한 출력층(Dense) 가중치의 표준편차는 약 0.006이다. Adam lr 1e-3이면 한 번에 최대 약 0.001(그 17%), 1e-2면 약 0.01(170%)씩 움직인다. 출력층이 전체 파라미터의 97%(633,611개)라서 이 층이 학습률에 가장 먼저 반응할 것으로 본다 (추정).

## 실행 방법

위에서부터 셀을 차례로 돌린다 (셀 0 → 1 → 2 → 3 → 4).

- 실행이 하나 끝날 때마다 결과가 Drive의 `food11_v2_lr.csv`에 한 줄씩 쌓인다. **런타임이 끊겨도 앞선 결과는 남는다.** 셀 0·1부터 다시 돌리면 끝난 실행은 건너뛰고 이어서 한다.
- 처음부터 다시 하려면 Drive에서 그 CSV 파일을 지운다.
- 셀 2(Keras)와 셀 3(PyTorch)은 서로 독립이다. 셀 4(요약)는 있는 결과만으로 만든다 (Keras만 먼저 돌려서 봐도 된다).
- 실행 하나마다 **시작 줄 → 에폭 줄 5개 → 끝 줄**이 출력된다. 에폭 줄은 Keras 약 30초, PyTorch 약 55초마다 올라오고 train·val 정확도(와 loss)가 찍힌다. 끝 줄에는 test와 걸린 시간이 붙는다.
- 큰 학습률(1e-2, 1e-1)에서 에폭 줄에 `nan`이 찍히고 끝 줄에 `발산`이 붙는 건 오류가 아니라 결과다.
- 시간이 더 있으면 셀 1의 `SEEDS`를 `(0, 1, 2)`로 바꿔서 셀 1부터 다시 돌린다. 시드 0은 건너뛰고 1·2만 추가로 돈다 (시간은 시드 수에 비례).
- **셀 4의 출력을 통째로 붙여 주면 된다.**

## 어떻게 읽을까

셀 4는 출력 4개와 그래프 2개를 만든다: `[결과표] Keras` · `[결과표] PyTorch` · `[최적 lr 후보]` · `[기준선 확인]` · 그래프 `lr_overview.png`(학습률 대 결과)와 `lr_curves.png`(에폭별 곡선).

1. **결과표**에서 옵티마이저마다 학습률이 커질 때 train acc가 어떻게 변하는지(학습 속도)를 보고, 그다음 val loss 최저와 그 에폭을 본다.
2. **최적 lr 후보**에서 옵티마이저마다 val 기준으로 가장 좋은 학습률을 본다. 고르는 기준은 val이고 test는 보고용이다 (v1의 선택 편향).
3. **기준선 확인**: Adam · 1e-3 · 시드 0이 1단계 시드 0 결과와 비슷해야 한다. 크게 벗어나면 이 노트북의 배선을 의심한다.

읽을 때 주의할 것

- **시드 하나라서 작은 차이는 읽지 않는다.** 같은 조건(He 기준선)의 test acc가 시드에 따라 1.2~3.4%p 흔들렸고, 같은 시드로 다시 돌려도 약 1.5%p 흔들렸다. 이 단계에서는 학습률이 10배 달라질 때 나타나는 큰 변화(train acc 수십 %p, val loss가 찍기 수준 2.4까지 오름 등)만 믿는다. 가까운 후보끼리의 우열(예: val acc 1%p)은 결론 내지 않고 다음 단계에서 시드 3개로 가린다.
- val loss 최저와 val acc 최고가 서로 다른 학습률을 가리키면 둘 다 후보로 남긴다. 목적이 정확도이긴 하지만 loss 0.006 차이로 정확도 3.5%p를 잃은 적이 v1에 있다 (HANDOFF 6번).
- **격자 끝**: 최적이 가장 작은/큰 학습률이면 진짜 최적은 그 바깥에 있을 수 있다.
- **최저 에폭 = 5**: 5에폭이 끝날 때도 val loss가 내려가는 중이었다는 뜻이다. 이 학습률은 5에폭 기준으로 불리하게 평가된 것일 수 있다 (더 길게 돌리면 달라질 수 있다 — 미탐색 항목).
- **발산**: 손실이 NaN이 된 실행. 이 실행의 정확도 숫자는 의미가 없다.
- 그래프의 점선은 찍기 수준이다 (정확도 1/11 ≈ 9%, 손실 ln 11 ≈ 2.40). 속이 빈 표시는 y축 한계(손실 3.0)를 넘어 잘린 값이다.

### 예상 (가설 — 결과로 확인한다)

1. **Adam**은 1e-3 근처가 가장 좋고 양쪽으로 나빠진다. 1e-4는 느려서 val loss 최저가 4~5에폭에 찍히고(아직 내려가는 중), 1e-2 이상은 눈에 띄게 나빠진다.
2. **SGD + momentum**의 최적은 Adam보다 한 자릿수 위, 1e-2 근처다. 1e-4는 거의 못 배우고 1e-1은 불안정하다.
3. **SGD**는 1e-3 이하에서 거의 못 배운다 (v1에서 20.5%). 최적이 격자 맨 위(1e-1)에 있을 가능성이 높다 — 그러면 "격자 끝" 경고가 나온다.
4. 따라서 최적 학습률은 **SGD > SGD+momentum > Adam** 순이고 간격은 10배 안팎이다. "전부 1e-3"이 공정하지 않았다는 게 확인된다.
5. **val loss 최저값은 잘 배우는 학습률끼리 비슷할 것이다 (1단계처럼 1.7~1.8).** 학습률은 그 값에 닿는 *속도*를 바꾸지 닿는 *한계*를 바꾸지 못한다는 가설이다. 틀려서 최저값이 학습률에 따라 뚜렷하게 달라지면, 학습률이 일반화에도 영향을 준다는 뜻이다.
6. 두 프레임워크에서 같은 방향이 나올 것이다. 다르다면 가장 큰 학습률(1e-2, 1e-1)에서 먼저 나타날 것이다 — Keras는 모델 끝에 softmax가 있고 PyTorch는 logit을 그대로 손실에 넣어서 수치 안정성이 다를 수 있다 (남은 통일 항목 중 '출력 형태').

### 기준선 칸 (Adam · 1e-3 · 시드 0)의 1단계 값

| | train acc | val acc | test acc |
|---|---|---|---|
| Keras | 77.3% | 43.1% | 46.3% |
| PyTorch | 66.3% | 41.1% | 42.9% |

## 셀 0 — 데이터 복사 (최초 1회, 런타임 끊기면 다시)

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

# Drive에서 직접 읽으면 학습이 수십 배 느려져서 로컬로 복사한다
if not os.path.isdir("/content/food11"):
    !cp -r /content/drive/MyDrive/food11 /content/food11

## 셀 1 — 공통 설정과 기록 함수

In [ ]:
import csv
import gc
import json
import os
import time

import numpy as np

DATA_DIR = "/content/food11"
IMG, BATCH, EPOCHS = 128, 32, 5
NUM_CLASSES = 11
N_PARAMS = 653_003           # 두 프레임워크 모두 이 값이어야 같은 모델이다

# 이 단계에서 바꾸는 것은 옵티마이저와 학습률뿐이다.
# 나머지(He 초기화, 구조, batch 32, 5 epoch, 증강 없음)는 1단계 그대로다.
OPTIMIZERS = ("adam", "sgd_momentum", "sgd")    # sgd_momentum = SGD + momentum 0.9
LRS = (1e-4, 1e-3, 1e-2, 1e-1)
SEEDS = (0,)                 # (0, 1, 2)로 바꾸면 시드 0은 건너뛰고 1·2만 추가로 돈다

# 실행 하나가 끝날 때마다 한 줄씩 쌓는다. 런타임이 끊겨도 남고,
# 다시 돌리면 끝난 실행은 건너뛴다. 처음부터 다시 하려면 이 파일을 지운다.
RESULTS = "/content/drive/MyDrive/food11_v2_lr.csv"
FIELDS = ["framework", "optimizer", "lr", "seed", "n_params", "minutes",
          "train_acc", "train_loss", "val_acc", "val_loss",    # 에폭별 값 (JSON 리스트)
          "test_acc", "test_loss"]


def lr_str(lr):
    return f"{lr:.0e}".replace("e-0", "e-")      # 0.0001 -> 1e-4


def done_runs():
    if not os.path.exists(RESULTS):
        return set()
    with open(RESULTS, newline="", encoding="utf-8") as f:
        return {(r["framework"], r["optimizer"], float(r["lr"]), int(r["seed"]))
                for r in csv.DictReader(f)}


def save_row(row):
    is_new = not os.path.exists(RESULTS)
    with open(RESULTS, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if is_new:
            w.writeheader()
        w.writerow(row)


def make_row(framework, optimizer, lr, seed, n_params, minutes, hist, test_loss, test_acc):
    return dict(framework=framework, optimizer=optimizer, lr=lr, seed=seed,
                n_params=n_params, minutes=round(minutes, 2),
                train_acc=json.dumps(hist["train_acc"]),
                train_loss=json.dumps(hist["train_loss"]),
                val_acc=json.dumps(hist["val_acc"]),
                val_loss=json.dumps(hist["val_loss"]),
                test_acc=test_acc, test_loss=test_loss)


def tag(framework, optimizer, lr, seed):
    return f"[{framework} · {optimizer} · lr {lr_str(lr)} · 시드 {seed}]"


def report(row):
    tr = json.loads(row["train_acc"])[-1]
    va = json.loads(row["val_acc"])[-1]
    diverged = not np.isfinite(json.loads(row["val_loss"])[-1])
    print(f"{tag(row['framework'], row['optimizer'], float(row['lr']), row['seed'])} "
          f"train {tr:.3f} · val {va:.3f} · test {float(row['test_acc']):.3f} "
          f"· {row['minutes']:.1f}분" + (" · 발산(loss NaN)" if diverged else ""))

## 셀 2 — Keras 12번 (옵티마이저 3 × 학습률 4) · 약 30분

In [ ]:
import keras
from keras import layers


def make_optimizer(name, lr):
    if name == "adam":
        return keras.optimizers.Adam(learning_rate=lr)
    if name == "sgd_momentum":
        return keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    return keras.optimizers.SGD(learning_rate=lr)


def show_epoch(epoch, logs):
    print(f"  에폭 {epoch + 1}/{EPOCHS}  "
          f"train {logs['accuracy']:.3f} (loss {logs['loss']:.3f})  ·  "
          f"val {logs['val_accuracy']:.3f} (loss {logs['val_loss']:.3f})", flush=True)


def run_keras(opt_name, lr, seed):
    keras.utils.set_random_seed(seed)

    def load(split, shuffle):
        ds = keras.utils.image_dataset_from_directory(
            f"{DATA_DIR}/{split}", image_size=(IMG, IMG),
            batch_size=BATCH, shuffle=shuffle, seed=seed)
        rescale = layers.Rescaling(1.0 / 255)
        return ds.map(lambda x, y: (rescale(x), y))

    train_ds = load("training", True)
    val_ds = load("validation", False)
    test_ds = load("evaluation", False)

    # 128 -> conv 126 -> pool 63 -> conv 61 -> pool 30
    model = keras.Sequential([
        keras.Input(shape=(IMG, IMG, 3)),

        layers.Conv2D(32, 3, activation="relu", kernel_initializer="he_normal"),
        layers.MaxPooling2D(2),

        layers.Conv2D(64, 3, activation="relu", kernel_initializer="he_normal"),
        layers.MaxPooling2D(2),

        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation="softmax",
                     kernel_initializer="he_normal"),
    ])
    n_params = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    assert n_params == N_PARAMS, n_params

    model.compile(
        optimizer=make_optimizer(opt_name, lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    t0 = time.time()
    h = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=0,
                  callbacks=[keras.callbacks.LambdaCallback(on_epoch_end=show_epoch)]).history
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    hist = {"train_acc": h["accuracy"], "train_loss": h["loss"],
            "val_acc": h["val_accuracy"], "val_loss": h["val_loss"]}
    row = make_row("keras", opt_name, lr, seed, n_params, (time.time() - t0) / 60,
                   {k: [float(v) for v in vs] for k, vs in hist.items()},
                   float(test_loss), float(test_acc))
    keras.backend.clear_session()
    gc.collect()
    return row


for seed in SEEDS:
    for opt_name in OPTIMIZERS:
        for lr in LRS:
            if ("keras", opt_name, lr, seed) in done_runs():
                print(f"{tag('keras', opt_name, lr, seed)} 이미 있음 — 건너뜀")
                continue
            print(f"{tag('keras', opt_name, lr, seed)} 시작 (약 2.5분)", flush=True)
            row = run_keras(opt_name, lr, seed)
            save_row(row)
            report(row)

## 셀 3 — PyTorch 12번 (옵티마이저 3 × 학습률 4) · 약 54분

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

tf_ = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),          # 0~1 변환 + HWC -> CHW
])


def make_optimizer(name, params, lr):
    if name == "adam":
        return torch.optim.Adam(params, lr=lr)
    if name == "sgd_momentum":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    return torch.optim.SGD(params, lr=lr)


def run_torch(opt_name, lr, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    g = torch.Generator().manual_seed(seed)

    def load(split, shuffle):
        ds = datasets.ImageFolder(f"{DATA_DIR}/{split}", transform=tf_)
        return DataLoader(ds, batch_size=BATCH, shuffle=shuffle,
                          generator=g if shuffle else None,
                          num_workers=2, pin_memory=True)

    train_dl = load("training", True)
    val_dl = load("validation", False)
    test_dl = load("evaluation", False)

    # 128 -> conv 126 -> pool 63 -> conv 61 -> pool 30
    model = nn.Sequential(
        nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Flatten(),
        nn.Dropout(0.5),
        nn.Linear(64 * 30 * 30, NUM_CLASSES),
    ).to(device)

    for m in model:       # 1단계 — He 초기화. 가중치만, bias는 PyTorch 기본값 그대로
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert n_params == N_PARAMS, n_params

    loss_fn = nn.CrossEntropyLoss()
    optimizer = make_optimizer(opt_name, model.parameters(), lr)

    def evaluate(dataloader):
        model.eval()
        total, correct, n = 0.0, 0, 0
        with torch.no_grad():
            for X, y in dataloader:
                X, y = X.to(device), y.to(device)
                pred = model(X)
                total += loss_fn(pred, y).item() * y.size(0)
                correct += (pred.argmax(1) == y).sum().item()
                n += y.size(0)
        return total / n, correct / n

    hist = {"train_acc": [], "train_loss": [], "val_acc": [], "val_loss": []}
    t0 = time.time()
    for epoch in range(EPOCHS):
        model.train()
        total, correct, n = 0.0, 0, 0
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total += loss.item() * y.size(0)
            correct += (pred.argmax(1) == y).sum().item()
            n += y.size(0)
        val_loss, val_acc = evaluate(val_dl)
        hist["train_acc"].append(correct / n)
        hist["train_loss"].append(total / n)
        hist["val_acc"].append(val_acc)
        hist["val_loss"].append(val_loss)
        print(f"  에폭 {epoch + 1}/{EPOCHS}  "
              f"train {hist['train_acc'][-1]:.3f} (loss {hist['train_loss'][-1]:.3f})  ·  "
              f"val {val_acc:.3f} (loss {val_loss:.3f})", flush=True)

    test_loss, test_acc = evaluate(test_dl)
    row = make_row("torch", opt_name, lr, seed, n_params, (time.time() - t0) / 60,
                   hist, test_loss, test_acc)
    del model, optimizer, train_dl, val_dl, test_dl
    torch.cuda.empty_cache()
    gc.collect()
    return row


for seed in SEEDS:
    for opt_name in OPTIMIZERS:
        for lr in LRS:
            if ("torch", opt_name, lr, seed) in done_runs():
                print(f"{tag('torch', opt_name, lr, seed)} 이미 있음 — 건너뜀")
                continue
            print(f"{tag('torch', opt_name, lr, seed)} 시작 (약 4.5분)", flush=True)
            row = run_torch(opt_name, lr, seed)
            save_row(row)
            report(row)

## 셀 4 — 요약 (이 셀의 출력을 붙여 주면 된다)

In [ ]:
import unicodedata
import warnings

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

with open(RESULTS, newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
assert rows, "결과가 아직 없다 — 셀 2·3을 먼저 돌릴 것"
assert all(int(r["n_params"]) == N_PARAMS for r in rows), "파라미터 수가 다른 실행이 있다"

FW = {"keras": "Keras", "torch": "PyTorch"}
OPT_NAME = {"adam": "Adam", "sgd_momentum": "SGD+momentum", "sgd": "SGD"}
CAP = 3.0                                     # 손실 그래프의 위쪽 한계
CHANCE_ACC = 100 / NUM_CLASSES                # 찍기 수준 정확도 (%)
CHANCE_LOSS = float(np.log(NUM_CLASSES))      # 찍기 수준 손실 (모든 클래스에 같은 확률)


def width(s):
    return sum(2 if unicodedata.east_asian_width(c) in "WF" else 1 for c in s)


def pad(s, w):
    return s + " " * max(0, w - width(s))


def bar(title):
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)


def metrics(r):
    ta, tl = np.array(json.loads(r["train_acc"])), np.array(json.loads(r["train_loss"]))
    va, vl = np.array(json.loads(r["val_acc"])), np.array(json.loads(r["val_loss"]))
    ok = np.isfinite(vl)
    best = int(np.argmin(np.where(ok, vl, np.inf)))
    return dict(train_acc=ta[-1] * 100, val_acc=va[-1] * 100,
                best_val_loss=float(vl[best]) if ok.any() else np.nan,
                best_ep=best + 1 if ok.any() else np.nan,
                test_acc=float(r["test_acc"]) * 100, test_loss=float(r["test_loss"]),
                diverged=bool(not np.isfinite(vl[-1]) or not np.isfinite(tl[-1])))


runs = {}      # (프레임워크, 옵티마이저, 학습률) -> {시드: 지표}
curves = {}    # (프레임워크, 옵티마이저, 학습률) -> {시드: 에폭별 곡선}
for r in rows:
    k = (r["framework"], r["optimizer"], float(r["lr"]))
    s = int(r["seed"])
    runs.setdefault(k, {})[s] = metrics(r)
    curves.setdefault(k, {})[s] = dict(
        train_acc=np.array(json.loads(r["train_acc"])) * 100,
        val_loss=np.array(json.loads(r["val_loss"])))

fws = [fw for fw in FW if any(k[0] == fw for k in runs)]
opts = [o for o in OPT_NAME if any(k[1] == o for k in runs)]
lrs = sorted({k[2] for k in runs})
seeds_all = sorted({s for g in runs.values() for s in g})
nseed = max(len(g) for g in runs.values())


def vals(group, key):
    v = np.array([m[key] for m in group.values()], dtype=float)
    return v[np.isfinite(v)]


def span(group, key, nd):
    v = vals(group, key)
    if len(v) == 0:
        return "-"
    if len(group) == 1:
        return f"{v[0]:.{nd}f}"
    return f"{v.mean():.{nd}f} [{v.min():.{nd}f}~{v.max():.{nd}f}]"


def mean_of(group, key):
    v = vals(group, key)
    return float(v.mean()) if len(v) else np.nan


def notes(group):
    out = []
    n_div = sum(m["diverged"] for m in group.values())
    if n_div:
        out.append(f"발산 {n_div}/{len(group)}")
    live = [m for m in group.values() if not m["diverged"]]
    if live and all(m["best_ep"] == EPOCHS for m in live):
        out.append("최저 에폭=마지막")
    return " · ".join(out)


# ---- 결과표 -----------------------------------------------------------------
COLS = [("train_acc", "train acc %", 1), ("val_acc", "val acc %", 1),
        ("best_val_loss", "val loss 최저", 2), ("best_ep", "최저 에폭", 0 if nseed == 1 else 1),
        ("test_acc", "test acc %", 1)]
CW = 15 if nseed == 1 else 22

for fw in fws:
    bar(f"[결과표] {FW[fw]} — 5에폭 끝 값 (시드 {', '.join(map(str, seeds_all))})"
        + ("" if nseed == 1 else ", 평균 [최소~최대]"))
    print(pad("옵티마이저 · lr", 22) + "".join(pad(name, CW) for _, name, _ in COLS) + "비고")
    for o in opts:
        for lr in lrs:
            g = runs.get((fw, o, lr))
            if not g:
                continue
            print(pad(f"{OPT_NAME[o]} · {lr_str(lr)}", 22)
                  + "".join(pad(span(g, key, nd), CW) for key, _, nd in COLS) + notes(g))
        print()

# ---- 최적 lr 후보 -----------------------------------------------------------
bar("[최적 lr 후보] 옵티마이저마다 val 기준으로 가장 좋은 학습률 (test는 보고용이라 쓰지 않는다)")
best = {}      # (프레임워크, 옵티마이저) -> (val loss 기준 lr, val acc 기준 lr)
for fw in fws:
    for o in opts:
        cand = {lr: runs[(fw, o, lr)] for lr in lrs
                if (fw, o, lr) in runs
                and not any(m["diverged"] for m in runs[(fw, o, lr)].values())}
        head = pad(FW[fw], 9) + pad(OPT_NAME[o], 14)
        if not cand:
            print(head + "고를 수 없음 (전부 발산했거나 결과가 없다)")
            continue
        by_loss = min(cand, key=lambda lr: mean_of(cand[lr], "best_val_loss"))
        by_acc = max(cand, key=lambda lr: mean_of(cand[lr], "val_acc"))
        best[(fw, o)] = (by_loss, by_acc)
        print(head
              + f"val loss 최저 기준 {lr_str(by_loss)} "
              + f"({mean_of(cand[by_loss], 'best_val_loss'):.2f}, "
              + f"{mean_of(cand[by_loss], 'best_ep'):.0f}에폭)"
              + f"  ·  val acc 기준 {lr_str(by_acc)} "
              + f"({mean_of(cand[by_acc], 'val_acc'):.1f}%)")
        for lr in sorted({by_loss, by_acc}):
            if lr in (lrs[0], lrs[-1]) and len(lrs) > 1:
                print(" " * 23 + f"! {lr_str(lr)}은 격자 끝 — 진짜 최적은 더 바깥에 있을 수 있다")
        if mean_of(cand[by_loss], "best_ep") >= EPOCHS:
            print(" " * 23 + "! val loss가 마지막 에폭에 최저 — 5에폭이 끝날 때도 내려가는 중이었다")


def agree(a, b):
    return "일치" if a == b else f"다름 (Keras {lr_str(a)} · PyTorch {lr_str(b)})"


if len(fws) == 2:
    print()
    for o in opts:
        a, b = best.get(("keras", o)), best.get(("torch", o))
        if a and b:
            print(pad(f"{OPT_NAME[o]} 두 프레임워크", 30)
                  + f"val loss 기준 {agree(a[0], b[0])}  ·  val acc 기준 {agree(a[1], b[1])}")

# ---- 기준선 확인 ------------------------------------------------------------
PREV = {   # 1단계(He) 노트북에서 시드 0으로 돌렸을 때 (%, 손실)
    "keras": dict(train_acc=77.29, test_acc=46.3, test_loss=1.8589),
    "torch": dict(train_acc=66.31, test_acc=42.90, test_loss=1.8580),
}
bar("[기준선 확인] Adam · 1e-3 · 시드 0 은 1단계(He) 시드 0과 거의 같아야 한다\n"
    "   같은 시드를 다시 돌렸을 때 나온 차이: train acc -0.7~+1.8%p, test acc -0.8~+1.4%p (GPU 비결정성)")
for fw in fws:
    g = runs.get((fw, "adam", 1e-3), {})
    if 0 not in g:
        print(f"{FW[fw]}: Adam · 1e-3 · 시드 0 결과가 없다")
        continue
    m, p = g[0], PREV[fw]
    d_tr, d_te = m["train_acc"] - p["train_acc"], m["test_acc"] - p["test_acc"]
    ok = abs(d_tr) <= 2.5 and abs(d_te) <= 2.0
    print(pad(FW[fw], 9)
          + f"train acc {m['train_acc']:.1f}% (1단계 {p['train_acc']:.1f}%, {d_tr:+.1f}%p) · "
          + f"test acc {m['test_acc']:.1f}% (1단계 {p['test_acc']:.1f}%, {d_te:+.1f}%p) · "
          + f"test loss {m['test_loss']:.2f} (1단계 {p['test_loss']:.2f})")
    print(" " * 9 + ("-> 재실행 차이 범위 안 (정상)" if ok else
                     "-> 범위 밖 — 이 노트북의 초기화·옵티마이저 배선을 확인할 것"))

# ---- 그래프 -----------------------------------------------------------------
SURFACE, INK, INK2, MUTED, GRID, AXIS = ("#fcfcfb", "#0b0b0b", "#52514e",
                                         "#898781", "#e1e0d9", "#c3c2b7")
OPT_STYLE = {"adam": ("#2a78d6", "o"), "sgd_momentum": ("#eb6834", "s"), "sgd": ("#1baf7a", "^")}
BLUES = ["#86b6ef", "#6da7ec", "#5598e7", "#3987e5", "#2a78d6",
         "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False


def lr_colors(n):
    if n == 1:
        return [BLUES[3]]
    return [BLUES[i] for i in np.linspace(0, len(BLUES) - 1, n).round().astype(int)]


def style(ax):
    ax.set_facecolor(SURFACE)
    ax.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(AXIS)
    ax.tick_params(colors=MUTED, labelsize=9)


CHANCE_HANDLE = Line2D([0], [0], color=MUTED, lw=0.9, ls=(0, (4, 3)), label="chance level")
CLIP_HANDLE = Line2D([0], [0], color=MUTED, ls="none", marker="o", ms=6, mfc=SURFACE,
                     mec=MUTED, mew=1.4, label=f"off-scale (clipped at {CAP:g})")
DIV_HANDLE = Line2D([0], [0], color=INK, ls="none", marker="x", ms=7, mew=1.6,
                    label="diverged (loss NaN)")
any_div = any(m["diverged"] for g in runs.values() for m in g.values())
SEED_TXT = (f"seed {seeds_all[0]}" if nseed == 1 else
            f"mean over {nseed} seeds, bar = min~max")
CURVE_TXT = f"seed {seeds_all[0]}" if nseed == 1 else f"mean over {nseed} seeds"

# 그래프 1 — 학습률에 따른 결과 (옵티마이저별 선)
OVERVIEW = [("train_acc", "train acc (%) at epoch 5", False),
            ("val_acc", "val acc (%) at epoch 5", False),
            ("best_val_loss", "best val loss (min over epochs)", True)]
clipped = False
fig, axes = plt.subplots(len(fws), 3, figsize=(13, 3.7 * len(fws) + 0.9), dpi=150,
                         squeeze=False, facecolor=SURFACE)
for i, fw in enumerate(fws):
    for j, (key, name, is_loss) in enumerate(OVERVIEW):
        ax = axes[i][j]
        style(ax)
        ax.set_xscale("log")
        seen = []
        for o in opts:
            color, marker = OPT_STYLE[o]
            pts = []
            for lr in lrs:
                g = runs.get((fw, o, lr), {})
                v = vals(g, key)
                if len(v):
                    pts.append((lr, v.mean(), v.min(), v.max(),
                                any(m["diverged"] for m in g.values())))
            if not pts:
                continue
            xs, mid, lo, hi, dv = (np.array(c) for c in zip(*pts))
            top = CAP if is_loss else np.inf
            seen += list(np.minimum(lo, top)) + list(np.minimum(mid, top))
            if nseed > 1:
                ax.vlines(xs, np.minimum(lo, top), np.minimum(hi, top), color=color,
                          lw=2.2, alpha=0.45, zorder=2)
            ax.plot(xs, np.minimum(mid, top), color=color, lw=1.8, marker=marker, ms=6,
                    mec=SURFACE, mew=1.2, solid_capstyle="round", zorder=3)
            over = mid > top
            if over.any():
                clipped = True
                ax.plot(xs[over], np.full(over.sum(), top), ls="none", marker=marker, ms=6,
                        mfc=SURFACE, mec=color, mew=1.4, clip_on=False, zorder=4)
            dv = dv.astype(bool)
            if dv.any():
                ax.plot(xs[dv], np.minimum(mid, top)[dv], ls="none", marker="x", ms=8,
                        mew=1.6, color=INK, zorder=5)
        ref = CHANCE_LOSS if is_loss else CHANCE_ACC
        ax.axhline(ref, color=MUTED, lw=0.9, ls=(0, (4, 3)), zorder=1)
        if is_loss:
            ax.set_ylim(min(1.6, min(seen + [ref]) - 0.05) if seen else 1.6, CAP)
        else:
            ax.set_ylim(0, max(seen + [ref]) * 1.12 if seen else 100)
        ax.set_xticks(lrs)
        ax.set_xticklabels([lr_str(lr) for lr in lrs])
        ax.minorticks_off()
        ax.set_xlabel("learning rate (log scale)", fontsize=9, color=INK2)
        ax.set_title(f"{FW[fw]} · {name}", fontsize=10.5, color=INK, loc="left")
handles = [Line2D([0], [0], color=OPT_STYLE[o][0], marker=OPT_STYLE[o][1], lw=1.8, ms=6,
                  mec=SURFACE, mew=1.2, label=OPT_NAME[o]) for o in opts]
handles += [CHANCE_HANDLE] + ([CLIP_HANDLE] if clipped else []) + ([DIV_HANDLE] if any_div else [])
fig.text(0.005, 0.99, f"Learning-rate scan, He init, 5 epochs  -  {SEED_TXT}",
         fontsize=12, color=INK, va="top")
fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.0, 0.955),
           ncol=len(handles), frameon=False, fontsize=9, labelcolor=INK2)
fig.tight_layout(rect=(0, 0, 1, 0.91))
fig.savefig("lr_overview.png", facecolor=SURFACE)
plt.show()
print("[그래프] lr_overview.png")

# 그래프 2 — 에폭별 곡선 (선 하나 = 학습률 하나, 연한 색 = 작은 학습률)
CURVE_ROWS = [("train_acc", "train acc (%)"), ("val_loss", "val loss")]
fig, axes = plt.subplots(len(fws) * 2, len(opts), figsize=(4.3 * len(opts) + 0.4,
                         2.9 * len(fws) * 2 + 0.9), dpi=150, squeeze=False, facecolor=SURFACE)
ramp = lr_colors(len(lrs))
ep = np.arange(1, EPOCHS + 1)
for i, fw in enumerate(fws):
    for r_, (key, name) in enumerate(CURVE_ROWS):
        for j, o in enumerate(opts):
            ax = axes[i * 2 + r_][j]
            style(ax)
            seen = []
            for lr, color in zip(lrs, ramp):
                c = curves.get((fw, o, lr))
                if not c:
                    continue
                arr = np.array([v[key] for v in c.values()])
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", RuntimeWarning)
                    y = np.nanmean(arr, axis=0)
                seen += [v for v in y if np.isfinite(v)]
                ax.plot(ep, y, color=color, lw=1.8, marker="o", ms=5.5, mec=SURFACE,
                        mew=1.0, solid_capstyle="round", zorder=3)
            is_loss = key == "val_loss"
            ax.axhline(CHANCE_LOSS if is_loss else CHANCE_ACC, color=MUTED, lw=0.9,
                       ls=(0, (4, 3)), zorder=1)
            if is_loss:
                ax.set_ylim(min(1.6, min(seen + [CHANCE_LOSS]) - 0.05) if seen else 1.6, CAP)
            else:
                ax.set_ylim(0, max(seen + [CHANCE_ACC]) * 1.12 if seen else 100)
            ax.set_xticks(ep)
            ax.set_xlabel("epoch", fontsize=9, color=INK2)
            ax.set_title(f"{FW[fw]} · {OPT_NAME[o]} · {name}", fontsize=10, color=INK, loc="left")
handles = [Line2D([0], [0], color=c, lw=1.8, marker="o", ms=5.5, mec=SURFACE, mew=1.0,
                  label=f"lr {lr_str(lr)}") for lr, c in zip(lrs, ramp)]
fig.text(0.005, 0.992, f"Curves by learning rate (light = small, dark = large)  -  {CURVE_TXT}",
         fontsize=12, color=INK, va="top")
fig.legend(handles=handles + [CHANCE_HANDLE], loc="upper left", bbox_to_anchor=(0.0, 0.962),
           ncol=len(handles) + 1, frameon=False, fontsize=9, labelcolor=INK2)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.savefig("lr_curves.png", facecolor=SURFACE)
plt.show()
print("[그래프] lr_curves.png")